# A-part Colab Quick Start

This notebook verifies the finalized A-part handoff without mounting Google Drive or downloading the raw Tsinghua images.

You only need two local files:

1. `D_RQ2_A_handoff.zip`
2. `sift_eval_uf.npy`

Run each cell from top to bottom using the triangle button on the left. This notebook evaluates DEV only. It does not re-run the final HOLD evaluation.

In [ ]:
# Step 1: Upload the two files from your computer.
# Select D_RQ2_A_handoff.zip and sift_eval_uf.npy together.
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded))

In [ ]:
# Step 2: Unzip the A-to-D package.
from pathlib import Path
import shutil

content = Path('/content')
zip_files = sorted(content.glob('D_RQ2_A_handoff*.zip'))
if not zip_files:
    raise FileNotFoundError('D_RQ2_A_handoff.zip was not uploaded.')
package = content / 'D_RQ2_A_handoff'
if package.exists():
    shutil.rmtree(package)
shutil.unpack_archive(str(zip_files[0]), str(package))
print('Package extracted to:', package)

In [ ]:
# Step 3: Check that all required A-part files are present.
models = sorted((package / 'models').glob('*_linear_weights.npz'))
required = [
    package / 'code' / 'eval.py',
    package / 'evalset' / 'breed_map.csv',
    package / 'evalset' / 'split.csv',
    package / 'evalset' / 'eval_index.csv',
    package / 'evalset' / 'dna_UF.csv',
]
missing = [str(path) for path in required if not path.exists()]
if len(models) != 12:
    raise RuntimeError(f'Expected 12 checkpoints, found {len(models)}')
if missing:
    raise FileNotFoundError(f'Missing files: {missing}')
print('OK: 12 classifier checkpoints found')
print('OK: evaluation code and CSV files found')

In [ ]:
# Step 4: Run the SIFT DEV evaluation.
# Expected representative result: 19 scorable dogs and 0 Top-1 hits.
import subprocess

feature_path = content / 'sift_eval_uf.npy'
if not feature_path.exists():
    raise FileNotFoundError('sift_eval_uf.npy was not uploaded.')
result_dir = content / 'a_sift_dev_results'
weights = [
    package / 'models' / 'RQ1-sift-clf-s1_linear_weights.npz',
    package / 'models' / 'RQ1-sift-clf-s2_linear_weights.npz',
    package / 'models' / 'RQ1-sift-clf-s3_linear_weights.npz',
]
command = [
    'python', str(package / 'code' / 'eval.py'),
    '--features', str(feature_path),
    '--index', str(package / 'evalset' / 'eval_index.csv'),
    '--weights', *map(str, weights),
    '--dna', str(package / 'evalset' / 'dna_UF.csv'),
    '--breed-map', str(package / 'evalset' / 'breed_map.csv'),
    '--split', str(package / 'evalset' / 'split.csv'),
    '--target-split', 'dev',
    '--output-dir', str(result_dir),
]
subprocess.run(command, check=True)
print('DEV evaluation completed.')

In [ ]:
# Step 5: Display the result table.
import pandas as pd
summary = pd.read_csv(result_dir / 'summary.csv')
display(summary[['seed', 'total_dogs', 'scorable_dogs', 'coverage', 'top1_hits', 'top1_accuracy']])

## What success looks like

The final `ensemble` row should show:

- `total_dogs = 20`
- `scorable_dogs = 19`
- `coverage = 0.95`
- `top1_hits = 0`

Files uploaded directly to Colab disappear when the runtime resets. That is fine for this verification notebook.